# 🐶 Dog Emotion Classifier — CNN Training
**Classes:** Happy · Sad · Angry  
**Architecture:** MobileNetV2 (Transfer Learning) — ideal for small datasets (~100 imgs/class)

### Folder structure expected in Google Drive:
```
MyDrive/
  Dog_Emotions/
    Happy/   ← all happy dog images
    Sad/     ← all sad dog images
    Angry/   ← all angry dog images
```

In [ ]:
# ── CELL 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELL 2: Install extra packages ──────────────────────────────────────────
# (tensorflow is pre-installed in Colab)
!pip install -q scikit-learn matplotlib seaborn

In [ ]:
# ── CELL 3: Imports ──────────────────────────────────────────────────────────
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# ── CELL 4: Config ────────────────────────────────────────────────────────────
DATASET_PATH = "/content/drive/MyDrive/Dog_Emotions"  # ← change if needed
IMG_SIZE     = (224, 224)   # MobileNetV2 native size
BATCH_SIZE   = 16           # small batch for small dataset
EPOCHS_FROZEN   = 15        # train only the head first
EPOCHS_FINETUNE = 20        # then fine-tune top layers of base
SEED         = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
# ── CELL 5: Data Generators with Augmentation ────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    # augmentation (only applied to training set)
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=SEED
)

val_data = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=SEED
)

CLASS_NAMES = list(train_data.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", train_data.class_indices)
print(f"Training samples  : {train_data.samples}")
print(f"Validation samples: {val_data.samples}")

In [ ]:
# ── CELL 6: Preview augmented samples ────────────────────────────────────────
imgs, labels = next(train_data)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i])
    ax.set_title(CLASS_NAMES[np.argmax(labels[i])], fontsize=10)
    ax.axis('off')
plt.suptitle("Augmented Training Samples", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── CELL 7: Build Model (MobileNetV2 Transfer Learning) ──────────────────────
base_model = MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False   # freeze base initially

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

In [ ]:
# ── CELL 8: Phase 1 — Train the head (base frozen) ───────────────────────────
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1 = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-6, monitor='val_loss'),
    ModelCheckpoint('best_model_phase1.h5', save_best_only=True, monitor='val_accuracy')
]

history1 = model.fit(
    train_data,
    epochs=EPOCHS_FROZEN,
    validation_data=val_data,
    callbacks=callbacks_phase1
)

print(f"\nPhase 1 best val accuracy: {max(history1.history['val_accuracy']):.4f}")

In [ ]:
# ── CELL 9: Phase 2 — Fine-tune top layers of MobileNetV2 ────────────────────
# Unfreeze top 30 layers of base model for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),   # lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    EarlyStopping(patience=7, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(patience=3, factor=0.3, min_lr=1e-8, monitor='val_loss'),
    ModelCheckpoint('best_model_phase2.h5', save_best_only=True, monitor='val_accuracy')
]

history2 = model.fit(
    train_data,
    epochs=EPOCHS_FINETUNE,
    validation_data=val_data,
    callbacks=callbacks_phase2
)

print(f"\nPhase 2 best val accuracy: {max(history2.history['val_accuracy']):.4f}")

In [ ]:
# ── CELL 10: Plot Training History ───────────────────────────────────────────
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

hist = merge_histories(history1, history2)
ep = range(1, len(hist['accuracy']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ep, hist['accuracy'],     label='Train Accuracy', color='steelblue')
ax1.plot(ep, hist['val_accuracy'], label='Val Accuracy',   color='orangered', linestyle='--')
ax1.axvline(EPOCHS_FROZEN, color='green', linestyle=':', label='Fine-tune start')
ax1.set_title('Model Accuracy', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep, hist['loss'],     label='Train Loss', color='steelblue')
ax2.plot(ep, hist['val_loss'], label='Val Loss',   color='orangered', linestyle='--')
ax2.axvline(EPOCHS_FROZEN, color='green', linestyle=':', label='Fine-tune start')
ax2.set_title('Model Loss', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 11: Evaluate + Confusion Matrix ─────────────────────────────────────
val_data.reset()
y_pred_probs = model.predict(val_data, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_data.classes

print("\n── Classification Report ──")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix', fontweight='bold')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CELL 12: Save Model + Class Indices ──────────────────────────────────────
MODEL_SAVE_PATH = 'dog_emotion_model.h5'
CLASS_INDICES_PATH = 'class_indices.json'

model.save(MODEL_SAVE_PATH)
with open(CLASS_INDICES_PATH, 'w') as f:
    json.dump(train_data.class_indices, f)

print(f"✅ Model saved to: {MODEL_SAVE_PATH}")
print(f"✅ Class indices saved to: {CLASS_INDICES_PATH}")
print(f"   Class mapping: {train_data.class_indices}")

In [ ]:
# ── CELL 13: Also copy to Google Drive (for persistence) ─────────────────────
import shutil

DRIVE_SAVE_DIR = "/content/drive/MyDrive/Dog_Emotions_Model"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

for fname in [MODEL_SAVE_PATH, CLASS_INDICES_PATH, 'training_history.png', 'confusion_matrix.png']:
    if os.path.exists(fname):
        shutil.copy(fname, DRIVE_SAVE_DIR)
        print(f"  Copied {fname} → Drive")

print("\n✅ All files backed up to Google Drive.")

In [ ]:
# ── CELL 14: Download files locally ──────────────────────────────────────────
from google.colab import files
files.download(MODEL_SAVE_PATH)
files.download(CLASS_INDICES_PATH)

---
## 📷 Quick Test — Upload a single image (optional)
Run this cell to test the trained model on any dog photo.

In [ ]:
# ── CELL 15: Quick single-image test ─────────────────────────────────────────
from google.colab import files
from tensorflow.keras.preprocessing import image as keras_image

uploaded = files.upload()
filename  = list(uploaded.keys())[0]

img       = keras_image.load_img(filename, target_size=IMG_SIZE)
img_array = keras_image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

probs          = model.predict(img_array)[0]
predicted_idx  = np.argmax(probs)
predicted_label = CLASS_NAMES[predicted_idx]
confidence     = probs[predicted_idx] * 100

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis('off')
plt.title(f"Prediction: {predicted_label}\nConfidence: {confidence:.1f}%",
          fontsize=13, fontweight='bold')
plt.show()

print("\nAll class probabilities:")
for name, prob in zip(CLASS_NAMES, probs):
    bar = '█' * int(prob * 30)
    print(f"  {name:<8} {bar:<30} {prob*100:.1f}%")